In [1]:
import pandas as pd

df = pd.read_csv("week10_students.csv")

print("Shape:", df.shape)   # expect (150, 9)

# Correlation of score with each candidate feature
corrs = df[["quiz_avg", "assignments_avg", "attendance_pct", "score"]].corr()["score"]
print(corrs.drop("score").round(3))

Shape: (150, 9)
quiz_avg           0.801
assignments_avg    0.536
attendance_pct     0.238
Name: score, dtype: float64


In [2]:
from sklearn.model_selection import train_test_split

features = ["attendance_pct", "quiz_avg", "assignments_avg"]
X = df[features]
y = df["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Train size:", X_train.shape[0])   # 112
print("Test size: ", X_test.shape[0])    # 38


Train size: 112
Test size:  38


In [3]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

# Each feature's coefficient with its name
for name, coef in zip(X_train.columns, model.coef_):
    print(f"{name:16s} {coef:+.3f}")
print(f"{'intercept':16s} {model.intercept_:+.3f}")


attendance_pct   +0.113
quiz_avg         +0.579
assignments_avg  +0.365
intercept        -4.428


In [4]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)   # TEST set only, never the training set

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f} points")
print(f"R2:  {r2:.2f}")

MAE: 3.33 points
R2:  0.89


In [5]:
import pandas as pd

# One-row DataFrame with the SAME column names (and order) used in training
new_student = pd.DataFrame({
    "attendance_pct":  [90],
    "quiz_avg":        [70],
    "assignments_avg": [75],
})

pred = model.predict(new_student)[0]
print(f"Predicted score for the new student: {pred:.1f}")

# Now my own student, with invented numbers
me = pd.DataFrame({
    "attendance_pct":  [85],
    "quiz_avg":        [68],
    "assignments_avg": [80],
})

my_pred = model.predict(me)[0]
print(f"Predicted score for me (invented numbers): {my_pred:.1f}")

Predicted score for the new student: 73.7
Predicted score for me (invented numbers): 73.8


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# (a) Target: 1 = passed (score >= 50), 0 = failed
df["passed"] = (df["score"] >= 50).astype(int)
print(df["passed"].value_counts())            # counts of 1 (pass) and 0 (fail)
print(df["passed"].value_counts().rename({1: "pass", 0: "fail"}))

# (b) Same features, new target, same split settings
yc = df["passed"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, yc, test_size=0.25, random_state=42
)

clf = LogisticRegression(max_iter=1000)
clf.fit(Xc_train, yc_train)

passed
1    131
0     19
Name: count, dtype: int64
passed
pass    131
fail     19
Name: count, dtype: int64


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [7]:
from sklearn.metrics import accuracy_score, confusion_matrix

yc_pred = clf.predict(Xc_test)

# (a) Model accuracy on the TEST set
print("Model accuracy:   ", round(accuracy_score(yc_test, yc_pred), 3))

# (b) Lazy baseline: predict "pass" for everyone.
# Its accuracy equals the share of passers in the test set.
print("Lazy baseline:    ", round(yc_test.mean(), 3))

# (c) Confusion matrix. Rows = actual, columns = predicted.
# Layout: [[TN, FP],
#          [FN, TP]]  where 0 = fail, 1 = pass
cm = confusion_matrix(yc_test, yc_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f"Failing students caught: {tn} of {tn + fp}")

Model accuracy:    0.895
Lazy baseline:     0.895
[[ 3  1]
 [ 3 31]]
Failing students caught: 3 of 4


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd

y = df["score"]

def eval_r2(X_features):
    """Same split settings every time; return test-set R2."""
    Xtr, Xte, ytr, yte = train_test_split(
        X_features, y, test_size=0.25, random_state=42
    )
    m = LinearRegression().fit(Xtr, ytr)
    return r2_score(yte, m.predict(Xte))

# (1) quiz_avg only (double brackets keep it a DataFrame)
X1 = df[["quiz_avg"]]

# (2) all three numeric features
num_features = ["attendance_pct", "quiz_avg", "assignments_avg"]
X2 = df[num_features]

# (3) three numeric features + program one-hot encoded
X3 = pd.concat(
    [df[num_features], pd.get_dummies(df["program"], drop_first=True)],
    axis=1
)

results = pd.DataFrame({
    "version": ["1. quiz_avg only", "2. three numeric", "3. three + program"],
    "n_features": [X1.shape[1], X2.shape[1], X3.shape[1]],
    "test_R2": [eval_r2(X1), eval_r2(X2), eval_r2(X3)],
})
print(results.round(3).to_string(index=False))

           version  n_features  test_R2
  1. quiz_avg only           1    0.624
  2. three numeric           3    0.887
3. three + program           6    0.880


In [10]:
import plotly.express as px

# Pass/fail as readable labels, so Plotly uses two distinct colours
# (a 0/1 number would give a continuous colour bar instead)
plot_df = df.copy()
plot_df["passed"] = plot_df["passed"].map({1: "Pass", 0: "Fail"})

# (a) Histogram of scores, 12 bins
fig1 = px.histogram(
    plot_df, x="score", nbins=12,
    title="1. Where the class stands",
    labels={"score": "Final score"}
)
fig1.show()

# (b) Scatter: quiz average vs score, coloured by pass/fail, name on hover
fig2 = px.scatter(
    plot_df, x="quiz_avg", y="score",
    color="passed", hover_name="name",
    title="2. Who is at risk",
    labels={"quiz_avg": "Quiz average", "score": "Final score"}
)
fig2.show()

# (c) Bar chart: average score per district
district_avg = (
    df.groupby("district", as_index=False)["score"].mean()
      .sort_values("score", ascending=False)
)
fig3 = px.bar(
    district_avg, x="district", y="score",
    title="3. Where to act",
    labels={"score": "Average score", "district": "District"}
)
fig3.show()

In [11]:
fig2.write_html("class_dashboard.html")
print("Saved class_dashboard.html")

Saved class_dashboard.html


In [12]:
fig4 = px.box(
    df, x="program", y="score", color="program",
    title="4. One chart, four stories: score spread by program"
)
fig4.show()

# Numbers to back up your two sentences
print(df.groupby("program")["score"].agg(["std", "min", "max"]).round(1))
print(df.groupby("program")["score"].quantile(0.75).sub(
      df.groupby("program")["score"].quantile(0.25)).round(1).rename("IQR"))

                       std   min   max
program                               
Accounting            11.0  43.0  89.5
Information Systems   12.3  28.6  84.4
Networking            10.8  41.9  89.4
Software Engineering  13.8  37.2  92.9
program
Accounting              16.6
Information Systems     17.7
Networking              13.9
Software Engineering    17.7
Name: IQR, dtype: float64


Model Card: Student Score Predictor

1. Purpose
This model predicts a student’s final score (0 to 100). It is for lecturers and coordinators, to spot students who may need help early.

2. Data and features
It uses 150 AUCA students. Three things go in: quiz average, assignments average and attendance. Quizzes matter most. The model learned from 75% of the students and was tested on the other 25%.

3. Performance

MAE ≈ 3.3: on new students, the prediction is usually about 3 marks away from the real score.
R² ≈ 0.89: the three inputs explain about 89% of why scores differ. The rest is things the model cannot see, like illness or a bad exam day.

4. Risks
A wrong prediction can make a good student look weak, or hide a student who needs help. The model only sees three numbers and was trained on one small group.

Use it to: find students who may need support, and then talk to them.
Do NOT use it to: decide grades, scholarships, admission or punishment. A person must always check before any action is taken.
